# Image Analyzer - Data Preprocessing & Model Training
Tento notebook provede 2 kroky projektu:
1. **MédiaPipe Extrakce (Data Preprocessing)** z obrázků (které jsi úspěšně nasbíral přes scraper).
2. **Natrénování Modelu (Machine Learning)** na vyextrahovaných X,Y,Z souřadnicích.

In [ ]:
# 1. Připojení Google Disku
# (Spusť tuto buňku, vyskočí okno s oprávněním pro připojení tvého G-Drive)
from google.colab import drive
drive.mount('/content/drive')

## Krok 1: Instalace a Rozbalení Fotek
Předpoklad: Na svůj Google Disk sis do kořenové složky nahrál ZIP soubor `raw_data.zip`, který obsahuje tvou lokální složku `raw` (s pod-složkami `middle_finger` a `other`).

In [ ]:
!pip install mediapipe opencv-python pandas numpy scikit-learn

import os
import zipfile

# Cesta k ZIPu na Google Disku (uprav, pokud se u tebe jmenuje jinak)
ZIP_PATH = '/content/drive/MyDrive/raw_data.zip'
EXTRACT_DIR = '/content/raw_data'

print("Rozbaluji data...")
with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)
print("Data úspešně rozbalena do", EXTRACT_DIR)

## Krok 2: Extrakce znaků (MediaPipe)
Tento krok projde ty tisíce rozbalených fotek a vytvoří obří tabulku `hand_landmarks.csv`.

In [ ]:
import cv2
import mediapipe as mp
import pandas as pd
import numpy as np

RAW_DIR = EXTRACT_DIR
if os.path.exists(os.path.join(EXTRACT_DIR, 'raw')):
    RAW_DIR = os.path.join(EXTRACT_DIR, 'raw')
    if os.path.exists(os.path.join(RAW_DIR, 'raw')):
        RAW_DIR = os.path.join(RAW_DIR, 'raw')

print(f"Budu cist data ze slozky: {RAW_DIR}")

OUTPUT_CSV = '/content/drive/MyDrive/hand_landmarks.csv'
CLASSES = {"other": 0, "middle_finger": 1}

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=True, max_num_hands=1, min_detection_confidence=0.5)

data_rows = []
images_scanned = 0
images_skipped = 0
images_success = 0

for class_name, label_id in CLASSES.items():
    class_folder = os.path.join(RAW_DIR, class_name)
    if not os.path.exists(class_folder):
        print(f"Nenalezena slozka {class_folder}")
        continue
        
    print(f"Zpracovávám třídu: {class_name}")
    for img_file in os.listdir(class_folder):
        if not img_file.lower().endswith(('.png', '.jpg', '.jpeg')):
            continue
            
        images_scanned += 1
        img_path = os.path.join(class_folder, img_file)
        image = cv2.imread(img_path)
        if image is None: continue
            
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = hands.process(image_rgb)
        
        if results.multi_hand_landmarks:
            images_success += 1
            hand_landmarks = results.multi_hand_landmarks[0]
            row = []
            for point in hand_landmarks.landmark:
                row.extend([point.x, point.y, point.z])
            row.append(label_id)
            data_rows.append(row)
        else:
            images_skipped += 1
        
        if images_scanned % 500 == 0:
            print(f"  Proskenováno {images_scanned} fotek...")

hands.close()

print("\n--- Přehled ---")
print(f"Skenováno fotek: {images_scanned}")
print(f"Z toho rukou nalezeno: {images_success}")
print(f"Odfiltrováno (žádná ruka): {images_skipped}")

if data_rows:
    columns = []
    for i in range(21):
        columns.extend([f"landmark_{i}_x", f"landmark_{i}_y", f"landmark_{i}_z"])
    columns.append("class_label")
    
    df = pd.DataFrame(data_rows, columns=columns)
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"✅ Data úspěšně uložena do {OUTPUT_CSV} na tvůj Google Disk!")

## Krok 3: Trénování Machine Learning Modelu (Random Forest)
Nyní z naší CSV tabulky vyrobíme chytrou umělou inteligenci.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

df = pd.read_csv(OUTPUT_CSV)
X = df.drop('class_label', axis=1)
y = df['class_label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Trénuji Random Forest model...")
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"\n✅ Model natrénován! Přesnost na neviděných datech (Test accuracy): {acc * 100:.2f}%")
print("\nReport klasifikace:")
print(classification_report(y_test, y_pred, target_names=['Ostatni Gesta (0)', 'Prostredniček (1)']))

MODEL_PATH = '/content/drive/MyDrive/middle_finger_model.pkl'
joblib.dump(model, MODEL_PATH)
print(f"\n💾 Náš natrénovaný model byl uložen na Google Drive do: {MODEL_PATH}")